# Lesson 2 — Your Beam on a Number Line

**Course:** Python for Structural Engineers — Simply Supported Beam  
**Prerequisites:** Lesson 1  
**You will learn:**
- Use lists and NumPy arrays to store a series of numbers
- Draw engineering diagrams with matplotlib
- Represent a parametric beam (total length, two support positions)
- Build an interactive widget to move supports and see the beam update live

**Time estimate:** 45–60 minutes

---

## The Beam We Are Analysing

Throughout this course we work with one beam. Its geometry is **parametric** — meaning the numbers are stored in variables so they can be changed easily.

```
x=0 ──[left cantilever]── x_A ──[interior span]── x_B ──[right cantilever]── x=L
                           PIN                      ROLLER

Default values: L = 10 m,  x_A = 2 m,  x_B = 8 m
```

- If `x_A = 0` → no left cantilever  
- If `x_B = L` → no right cantilever  

The entire beam is described by three numbers. Let's write them in Python:

In [ ]:
# Beam geometry
L_total = 10.0   # total beam length (m)
x_A     = 2.0    # position of pin support A from left end (m)
x_B     = 8.0    # position of roller support B from left end (m)

# Derived quantities
cantilever_left  = x_A
cantilever_right = L_total - x_B
interior_span    = x_B - x_A

print(f"Left cantilever  : {cantilever_left:.1f} m")
print(f"Interior span    : {interior_span:.1f} m")
print(f"Right cantilever : {cantilever_right:.1f} m")

---

## 2.1 NumPy Arrays — A Row of Numbers

When we plot the beam we need a long row of x-positions spread evenly along its length. **NumPy** (short for Numerical Python) is the standard library for this.

`np.linspace(start, stop, n)` creates `n` evenly-spaced numbers from `start` to `stop`:

In [ ]:
import numpy as np

# 11 evenly-spaced points from 0 to 10 m
x = np.linspace(0, L_total, 11)
print(x)

In [ ]:
# For smooth diagrams we'll use many more points
x_fine = np.linspace(0, L_total, 500)
print(f"Number of points : {len(x_fine)}")
print(f"First value      : {x_fine[0]} m")
print(f"Last value       : {x_fine[-1]} m")   # -1 means the last element

NumPy arrays support **element-wise arithmetic** — operations apply to every element at once:

In [ ]:
x_short = np.linspace(0, 10, 5)
print("x          :", x_short)
print("x * 2      :", x_short * 2)
print("x squared  :", x_short ** 2)

This will be very useful when we compute shear forces and moments at every point along the beam.

---

## 2.2 Drawing with Matplotlib

**Matplotlib** is Python's main plotting library. Let's draw the beam step by step.

In [ ]:
import matplotlib.pyplot as plt

# Step 1: create a figure and axes
fig, ax = plt.subplots(figsize=(12, 2.5))

# Step 2: draw the beam as a thick horizontal line
ax.plot([0, L_total], [0, 0], color="black", linewidth=5, solid_capstyle="round")

# Step 3: mark the support positions as vertical ticks
ax.plot([x_A, x_B], [0, 0], marker="|", markersize=20, color="gray", linestyle="none")

# Step 4: label key positions
for x_pos, label in [(0, "0"), (x_A, f"A\n{x_A} m"), (x_B, f"B\n{x_B} m"), (L_total, f"{L_total} m")]:
    ax.text(x_pos, -0.15, label, ha="center", va="top", fontsize=9)

ax.set_xlim(-0.5, L_total + 0.5)
ax.set_ylim(-0.5, 0.5)
ax.axis("off")
ax.set_title("Simply supported beam with cantilevers", fontsize=11)
plt.tight_layout()
plt.show()

Now let's add the engineering support symbols — a **pin** (solid triangle) and a **roller** (hollow triangle + circle):

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.5))

# ── Beam ──────────────────────────────────────────────────────────────────
ax.plot([0, L_total], [0, 0], color="black", linewidth=5, solid_capstyle="round", zorder=3)

# ── Cantilever shading ────────────────────────────────────────────────────
if x_A > 0:
    ax.axvspan(0, x_A, alpha=0.10, color="orange")          # left cantilever
if x_B < L_total:
    ax.axvspan(x_B, L_total, alpha=0.10, color="orange")    # right cantilever

# ── Pin support at A (solid triangle) ────────────────────────────────────
s = 0.18   # symbol half-size
pin = plt.Polygon([[x_A, 0], [x_A - s, -s*1.4], [x_A + s, -s*1.4]],
                  closed=True, facecolor="#888", edgecolor="black", linewidth=0.8, zorder=4)
ax.add_patch(pin)
ax.plot([x_A - s*1.3, x_A + s*1.3], [-s*1.4, -s*1.4], color="black", lw=1.5)

# ── Roller support at B (hollow triangle + circle) ────────────────────────
roller = plt.Polygon([[x_B, 0], [x_B - s, -s*1.4], [x_B + s, -s*1.4]],
                     closed=True, facecolor="white", edgecolor="black", linewidth=0.8, zorder=4)
ax.add_patch(roller)
circle = plt.Circle((x_B, -s*1.4 - s*0.55), s*0.5,
                    facecolor="white", edgecolor="black", linewidth=0.8, zorder=4)
ax.add_patch(circle)
ax.plot([x_B - s*1.3, x_B + s*1.3], [-s*1.4 - s*1.1, -s*1.4 - s*1.1],
        color="black", lw=1.5)

# ── Dimension annotations ─────────────────────────────────────────────────
ax.text(x_A, -0.55, f"A  ({x_A:.1f} m)", ha="center", fontsize=9, color="#444")
ax.text(x_B, -0.55, f"B  ({x_B:.1f} m)", ha="center", fontsize=9, color="#444")
ax.text(0,      0.08, "0",           ha="center", fontsize=8, color="#888")
ax.text(L_total, 0.08, f"{L_total} m", ha="center", fontsize=8, color="#888")

# ── Span label ────────────────────────────────────────────────────────────
mid = (x_A + x_B) / 2
ax.annotate("", xy=(x_B, 0.35), xytext=(x_A, 0.35),
            arrowprops=dict(arrowstyle="<->", color="#555", lw=1))
ax.text(mid, 0.42, f"Interior span = {interior_span:.1f} m", ha="center", fontsize=8, color="#555")

ax.set_xlim(-0.4, L_total + 0.4)
ax.set_ylim(-0.8, 0.7)
ax.axis("off")
ax.set_title(f"Beam: L = {L_total} m  |  Pin at {x_A} m  |  Roller at {x_B} m", fontsize=11)
plt.tight_layout()
plt.show()

---

## 2.3 Interactive Beam Widget

Let's wire up sliders so you can move the supports and change the span in real time.

Notice the structure: it's the same **5-step pattern** from Lesson 1.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

%matplotlib inline

# ── Step 1: Output area ───────────────────────────────────────────────────
out = widgets.Output()

# ── Step 2: Controls ──────────────────────────────────────────────────────
style  = {"description_width": "160px"}
layout = widgets.Layout(width="460px")

slider_L  = widgets.FloatSlider(value=10.0, min=5.0, max=20.0, step=0.5,
                                 description="Total length L (m):", style=style, layout=layout)
slider_xA = widgets.FloatSlider(value=2.0,  min=0.0, max=9.5,  step=0.5,
                                 description="Support A position (m):", style=style, layout=layout)
slider_xB = widgets.FloatSlider(value=8.0,  min=0.5, max=10.0, step=0.5,
                                 description="Support B position (m):", style=style, layout=layout)

# ── Step 3: Update function ───────────────────────────────────────────────
def draw_beam_widget(change=None):
    L   = slider_L.value
    x_A = slider_xA.value
    x_B = slider_xB.value

    # Enforce x_A < x_B with at least 0.5 m gap
    if x_A >= x_B - 0.5:
        x_A = x_B - 0.5
    # Enforce supports within beam
    x_B = min(x_B, L)

    cant_L = x_A
    cant_R = L - x_B
    span   = x_B - x_A

    with out:
        out.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(13, 3.8))

        # Beam
        ax.plot([0, L], [0, 0], color="black", linewidth=5,
                solid_capstyle="round", zorder=3)

        # Cantilever shading
        if cant_L > 0:
            ax.axvspan(0, x_A, alpha=0.10, color="#FF6F00",
                       label=f"Left cantilever ({cant_L:.1f} m)")
        if cant_R > 0:
            ax.axvspan(x_B, L, alpha=0.10, color="#FF6F00",
                       label=f"Right cantilever ({cant_R:.1f} m)")

        # Interior span
        ax.axvspan(x_A, x_B, alpha=0.05, color="#1565C0",
                   label=f"Interior span ({span:.1f} m)")

        # Pin (solid triangle)
        s = 0.18
        pin = plt.Polygon([[x_A, 0], [x_A-s, -s*1.4], [x_A+s, -s*1.4]],
                          closed=True, fc="#777", ec="black", lw=0.8, zorder=4)
        ax.add_patch(pin)
        ax.plot([x_A-s*1.3, x_A+s*1.3], [-s*1.4, -s*1.4], "k-", lw=1.5, zorder=4)

        # Roller (hollow triangle + circle)
        roller = plt.Polygon([[x_B, 0], [x_B-s, -s*1.4], [x_B+s, -s*1.4]],
                             closed=True, fc="white", ec="black", lw=0.8, zorder=4)
        ax.add_patch(roller)
        circ = plt.Circle((x_B, -s*1.4-s*0.55), s*0.5,
                          fc="white", ec="black", lw=0.8, zorder=4)
        ax.add_patch(circ)
        ax.plot([x_B-s*1.3, x_B+s*1.3],
                [-s*1.4-s*1.1, -s*1.4-s*1.1], "k-", lw=1.5, zorder=4)

        # Labels
        ax.text(x_A, -0.62, f"A ({x_A:.1f} m)", ha="center", fontsize=9, color="#333")
        ax.text(x_B, -0.62, f"B ({x_B:.1f} m)", ha="center", fontsize=9, color="#333")
        ax.text(0,    0.12, "0",        ha="center", fontsize=8, color="#888")
        ax.text(L,    0.12, f"{L:.1f} m", ha="center", fontsize=8, color="#888")

        # Span arrow
        mid = (x_A + x_B) / 2
        ax.annotate("", xy=(x_B, 0.42), xytext=(x_A, 0.42),
                    arrowprops=dict(arrowstyle="<->", color="#1565C0", lw=1))
        ax.text(mid, 0.50, f"Span = {span:.1f} m",
                ha="center", fontsize=8, color="#1565C0")

        ax.set_xlim(-0.4, L + 0.4)
        ax.set_ylim(-0.9, 0.75)
        ax.axis("off")
        ax.set_title(f"L = {L:.1f} m  |  Pin at A = {x_A:.1f} m  |  Roller at B = {x_B:.1f} m",
                     fontsize=11, pad=8)
        ax.legend(fontsize=8, loc="upper left", framealpha=0.8)
        plt.tight_layout()
        plt.show()

# ── Step 4: Connect sliders ───────────────────────────────────────────────
slider_L.observe(draw_beam_widget, names="value")
slider_xA.observe(draw_beam_widget, names="value")
slider_xB.observe(draw_beam_widget, names="value")

# ── Step 5: Initial render + display ─────────────────────────────────────
draw_beam_widget()
display(widgets.VBox([
    widgets.HTML("<b>Move the sliders to reshape the beam:</b>"),
    slider_L, slider_xA, slider_xB,
    out
]))

**Try these experiments:**

1. Move **Support A** all the way to `0` → the left cantilever disappears.
2. Move **Support B** all the way to the right edge → the right cantilever disappears.
3. Set A = 5 m, B = 5 m (they snap to 0.5 m apart) → a very short interior span with long cantilevers on both sides.

---

## 2.4 Practice Exercise

In the cell below, without the widget, calculate and print:
- The **length of each cantilever** for the default beam (L=10, x_A=2, x_B=8)
- The **midspan position** of the interior span (hint: `(x_A + x_B) / 2`)
- The **quarter-span position** from support A (hint: `x_A + interior_span / 4`)


In [ ]:
# Your code here
L_total = 10.0
x_A = 2.0
x_B = 8.0
interior_span = x_B - x_A


---

## Summary

**Python concepts covered:**
- `numpy` arrays with `np.linspace()` — a row of evenly-spaced numbers
- Array indexing: `x[0]` (first), `x[-1]` (last)
- `matplotlib.pyplot` — creating figures, plotting lines, adding text, patches
- `plt.Polygon`, `plt.Circle` — drawing shapes
- `ax.axvspan()` — shading a region between two x-values

**Structural concepts covered:**
- Beam geometry described by three parameters: L, x_A, x_B
- Pin support vs roller support — symbols and meaning
- Cantilever vs interior span

## Before the Next Lesson

In **Lesson 3** you'll add loads to the beam — point loads and uniformly distributed loads — using Python dictionaries and your first custom functions.